<a href="https://colab.research.google.com/github/YonaUz/RAG-and-LLM/blob/main/03_embeddings_and_vector_store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -U langchain-core langchain-openai langchain-chroma chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 8.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 98.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 98.0 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uni

In [1]:
import json
import os
from pathlib import Path
from getpass import getpass

from google.colab import userdata

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [2]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Satellite_Intelligence_Assistant"
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_FILE = PROCESSED_DATA_DIR / "split_documents.jsonl"

VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store" / "chroma_db"

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input file: {INPUT_FILE}")
print(f"Vector store directory: {VECTOR_STORE_DIR}")

assert INPUT_FILE.exists(), "The split documents file was not found."

Input file: /content/drive/MyDrive/Satellite_Intelligence_Assistant/data/processed/split_documents.jsonl
Vector store directory: /content/drive/MyDrive/Satellite_Intelligence_Assistant/data/vector_store/chroma_db


In [3]:
chunks = []
with INPUT_FILE.open("r", encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)

        chunk = Document(
            page_content=record["page_content"],
            metadata=record["metadata"],
        )
        chunks.append(chunk)
print(f"Loaded chunks: {len(chunks)}")

Loaded chunks: 698


In [4]:
print("First chunk preview:")
print(chunks[0].page_content[:500])

print("\nMetadata:")
print(chunks[0].metadata)

First chunk preview:
LSDS-1574 
Version 5.0 
Department of the Interior 
U.S. Geological Survey 
 
 
 
 
 
Landsat 8 (L8) 
Data Users Handbook 
 
 
 
 
 
 
 
 
 
 
 
 
 
Version 5.0 
 
November 2019

Metadata:
{'producer': 'Adobe PDF Library 19.21.79', 'creator': 'Acrobat PDFMaker 19 for Word', 'creationdate': '2019-11-19T10:06:32-06:00', 'author': 'aaker', 'company': 'EROS Data Center', 'moddate': '2019-11-26T15:30:49-06:00', 'sourcemodified': 'D:20191119160605', 'title': 'Landsat 8 Data Users Handbook', 'source': 'landsat_8_data_users_handbook.pdf', 'total_pages': 114, 'page': 0, 'page_label': '1', 'publisher': 'USGS', 'mission': 'Landsat 8', 'document_type': 'technical_handbook', 'url': 'https://www.usgs.gov/media/files/landsat-8-data-users-handbook', 'language': 'en', 'local_path': '/content/drive/MyDrive/Satellite_Intelligence_Assistant/data/raw/landsat/landsat_8_data_users_handbook.pdf', 'ingestion_date': '2026-11-09', 'chunk_index': 0, 'chunk_id': 'landsat_8_data_users_handbook.

In [5]:
try: # configure key on the left side of Colab notebook with name "OPENAI_API_KEY" and its value
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

except Exception:
    OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OpenAI API key configured.")

OpenAI API key configured.


In [6]:
EMBEDDING_MODEL_NAME = "text-embedding-3-small" # using an OpenAI model; requiring an API key

embedding_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME
)

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")

Embedding model: text-embedding-3-small


In [7]:
sample_text = (
    "Sentinel-2 Band 8 operates in the near-infrared region."
)
sample_embedding = embedding_model.embed_query(sample_text)

print(f"Embedding dimensions: {len(sample_embedding)}")
print(f"First five values: {sample_embedding[:5]}")

Embedding dimensions: 1536
First five values: [-0.0218658447265625, 0.0167999267578125, 0.0255126953125, -0.050933837890625, -0.0030345916748046875]


In [8]:
sample_texts = [
    "Sentinel-2 Band 8 operates in the near-infrared region.",
    "The near-infrared band of Sentinel-2 is B8.",
    "Landsat satellites have a specific revisit time.",
]

sample_embeddings = embedding_model.embed_documents(sample_texts)

print(f"Number of embeddings: {len(sample_embeddings)}") # each vector refers to 1 sentence
print(f"Dimensions per embedding: {len(sample_embeddings[0])}")

Number of embeddings: 3
Dimensions per embedding: 1536


In [9]:
import numpy as np

def cosine_similarity(vector_a, vector_b):
    """Calculate cosine similarity between two vectors."""

    vector_a = np.array(vector_a)
    vector_b = np.array(vector_b)

    return np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    )

similarity_1_2 = cosine_similarity(
    sample_embeddings[0],
    sample_embeddings[1],
)

similarity_1_3 = cosine_similarity(
    sample_embeddings[0],
    sample_embeddings[2],
)

print(f"Similarity between texts 1 and 2: {similarity_1_2:.4f}")
print(f"Similarity between texts 1 and 3: {similarity_1_3:.4f}")

Similarity between texts 1 and 2: 0.8431
Similarity between texts 1 and 3: 0.3537


In [10]:
chunk_ids = [
    chunk.metadata["chunk_id"]
    for chunk in chunks
]
assert len(chunk_ids) == len(set(chunk_ids)), (
    "Duplicate chunk IDs were found."
)
print(f"Unique chunk IDs: {len(chunk_ids)}")

Unique chunk IDs: 698


In [11]:
COLLECTION_NAME = "satellite_remote_sensing"

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(VECTOR_STORE_DIR),
)

vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids,
)

print("Stored chunks:", vector_store._collection.count())

Stored chunks: 698


In [12]:
collection_data = vector_store.get()
stored_ids = collection_data["ids"]
print(f"Stored chunks: {len(stored_ids)}")
print(f"Expected chunks: {len(chunks)}")

assert len(stored_ids) == len(chunks), (
    "The number of stored chunks does not match the input."
)

Stored chunks: 698
Expected chunks: 698


In [13]:
query = "What is the spatial resolution of Sentinel-2 Band 8?"
retrieved_chunks = vector_store.similarity_search(
    query,
    k=4,
)
print(f"Retrieved chunks: {len(retrieved_chunks)}")

Retrieved chunks: 4


In [16]:
for index, chunk in enumerate(retrieved_chunks):

    print(f"\n{'=' * 70}")
    print(f"Result {index + 1}")
    print(f"{'=' * 70}")

    print(f"Source: {chunk.metadata.get('source')}")
    print(f"Mission: {chunk.metadata.get('mission')}")
    print(f"Page: {chunk.metadata.get('page')}")
    print(f"Chunk ID: {chunk.metadata.get('chunk_id')}")

    print("\nContent:")
    print(chunk.page_content[:1000])



Result 1
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Page: 8
Chunk ID: sentinel_2_documentation.pdf_8_626

Content:
and atmospheric corrections (443 nm for aerosols, 945 nm for water 
vapour and 1375 nm for cirrus detection).  
 
The specified spectral band characteristics and resolutions are summarised in 
Figure 2-3.  
 
 
Figure 2-3. S2-MSI spectral bands and resolution. 
The spectral response average for Sentinel-2A and Sentinel-2B are given in Figure 
2-4.

Result 2
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Page: 8
Chunk ID: sentinel_2_documentation.pdf_8_625

Content:
Reference: S2-PDGS-MPC-ATBD-L1 
Issue:  1.1 
Date:   2023-01-31 
9/40 
 
 
           
 
 
Sentinel-2 MSI performs measurements in 13 spectral bands spread over the 
Visible to Near -Infrared (VNIR) and Short -Waves Infrared (SWIR) domains with 
spatial resolutions ranging from 10 to 60 m [REF-11]. 
These spectral channels include: 
• 4 bands at 10 m spatial resolution: blue (490 nm), gr

Let's analysis the retrieved chunks for the query _What is the spatial resolution of Sentinel-2 Band 8?_.
In the next cell, the distance score is added with `similarity_search_with_score`, firstly let's try to have an idea of how far are the chunks from the query by looking at those results (the first 1000 characters).
The first chunk result refers to Sentinel 2 documentation and refers to characteritics of the bands. It does not directly answer to the query, because it does not speak of spatial resolution and nor band 8, but sends back to the Figure 2-3, that we currently don't read.

The second result chunk is more relevant dealing with spatial resolution, and retrieves  information from the documentation of Sentinel 2. It still does not directly answer the question because, it enumerates the band names with the color and wavelenghts but not with band number. Then it has to make the match between the eigth band that it corresponds to the near infrared and conclude that the spatial resolution is 10m.

Then, chunk 3 still retrieves information from Sentinel documentation but not focusing on the subject to answer the query.

Finally chunk 4 deals with Landsat 8... and is far from answering the query.

However, the results enable us to identify the section, the source document.

In [44]:
scored_results = vector_store.similarity_search_with_score( # smaller the distance, the better the chunks is relevant
    query,
    k=4,
)

for index, (chunk, distance) in enumerate(scored_results):

    print(f"\nResult {index + 1}")
    print(f"Distance: {distance:.4f}")
    print(f"Source: {chunk.metadata.get('source')}")
    print(f"Mission: {chunk.metadata.get('mission')}")
    print(f"Page: {chunk.metadata.get('page')}")
    print(f"Content: {chunk.page_content[:400]}")


Result 1
Distance: 0.7695
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Page: 8
Content: and atmospheric corrections (443 nm for aerosols, 945 nm for water 
vapour and 1375 nm for cirrus detection).  
 
The specified spectral band characteristics and resolutions are summarised in 
Figure 2-3.  
 
 
Figure 2-3. S2-MSI spectral bands and resolution. 
The spectral response average for Sentinel-2A and Sentinel-2B are given in Figure 
2-4.

Result 2
Distance: 0.8454
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Page: 8
Content: Reference: S2-PDGS-MPC-ATBD-L1 
Issue:  1.1 
Date:   2023-01-31 
9/40 
 
 
           
 
 
Sentinel-2 MSI performs measurements in 13 spectral bands spread over the 
Visible to Near -Infrared (VNIR) and Short -Waves Infrared (SWIR) domains with 
spatial resolutions ranging from 10 to 60 m [REF-11]. 
These spectral channels include: 
• 4 bands at 10 m spatial resolution: blue (490 nm), green (560 n

Result 3
Distance: 0.9287
Source: sentinel_2_d

In [17]:
relevant_results = vector_store.similarity_search_with_relevance_scores(
    query,
    k=4,
)

for index, (chunk, distance) in enumerate(relevant_results):

    print(f"\nResult {index + 1}")
    print(f"Distance: {distance:.4f}")
    print(f"Source: {chunk.metadata.get('source')}")
    print(f"Mission: {chunk.metadata.get('mission')}")
    print(f"Page: {chunk.metadata.get('page')}")
    print(f"Content: {chunk.page_content[:400]}")


Result 1
Distance: 0.4559
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Page: 8
Content: and atmospheric corrections (443 nm for aerosols, 945 nm for water 
vapour and 1375 nm for cirrus detection).  
 
The specified spectral band characteristics and resolutions are summarised in 
Figure 2-3.  
 
 
Figure 2-3. S2-MSI spectral bands and resolution. 
The spectral response average for Sentinel-2A and Sentinel-2B are given in Figure 
2-4.

Result 2
Distance: 0.4023
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Page: 8
Content: Reference: S2-PDGS-MPC-ATBD-L1 
Issue:  1.1 
Date:   2023-01-31 
9/40 
 
 
           
 
 
Sentinel-2 MSI performs measurements in 13 spectral bands spread over the 
Visible to Near -Infrared (VNIR) and Short -Waves Infrared (SWIR) domains with 
spatial resolutions ranging from 10 to 60 m [REF-11]. 
These spectral channels include: 
• 4 bands at 10 m spatial resolution: blue (490 nm), green (560 n

Result 3
Distance: 0.3434
Source: sentinel_2_d

In [19]:
test_queries = [
    "What is the spatial resolution of Sentinel-2 Band 8?",
    "What is the difference between Landsat 8 OLI and TIRS sensors?",
    "What is the revisit time of Sentinel-2?",
    "What is the difference between Sentinel-2 Level-1C and Level-2A products?", # Level-2A is not covered in the data sheet
]

for query in test_queries:

    print(f"\n{'=' * 80}")
    print(f"QUERY: {query}")
    print(f"{'=' * 80}")

    results = vector_store.similarity_search(
        query,
        k=3,
    )

    for index, document in enumerate(results):

        print(f"\nResult {index + 1}")
        print(f"Source: {document.metadata.get('source')}")
        print(f"Mission: {document.metadata.get('mission')}")
        print(f"Page: {document.metadata.get('page')}")
        print(f"Content: {document.page_content[:350]}")


QUERY: What is the spatial resolution of Sentinel-2 Band 8?

Result 1
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Page: 8
Content: and atmospheric corrections (443 nm for aerosols, 945 nm for water 
vapour and 1375 nm for cirrus detection).  
 
The specified spectral band characteristics and resolutions are summarised in 
Figure 2-3.  
 
 
Figure 2-3. S2-MSI spectral bands and resolution. 
The spectral response average for Sentinel-2A and Sentinel-2B are given in Figure 
2-4.

Result 2
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Page: 8
Content: Reference: S2-PDGS-MPC-ATBD-L1 
Issue:  1.1 
Date:   2023-01-31 
9/40 
 
 
           
 
 
Sentinel-2 MSI performs measurements in 13 spectral bands spread over the 
Visible to Near -Infrared (VNIR) and Short -Waves Infrared (SWIR) domains with 
spatial resolutions ranging from 10 to 60 m [REF-11]. 
These spectral channels include: 
• 4 bands at 10

Result 3
Source: sentinel_2_documentation.pdf
Mission: Sentinel-2
Pag

We can filter retrievals by adding metadata filters

In [20]:
sentinel_results = vector_store.similarity_search(
    "What are the spectral bands of the satellite?",
    k=4,
    filter={"mission": "Sentinel-2"},
)

for index, document in enumerate(sentinel_results):

    print(f"\nResult {index + 1}")
    print(f"Mission: {document.metadata.get('mission')}")
    print(f"Source: {document.metadata.get('source')}")
    print(f"Content: {document.page_content[:400]}")


Result 1
Mission: Sentinel-2
Source: sentinel_2_documentation.pdf
Content: and atmospheric corrections (443 nm for aerosols, 945 nm for water 
vapour and 1375 nm for cirrus detection).  
 
The specified spectral band characteristics and resolutions are summarised in 
Figure 2-3.  
 
 
Figure 2-3. S2-MSI spectral bands and resolution. 
The spectral response average for Sentinel-2A and Sentinel-2B are given in Figure 
2-4.

Result 2
Mission: Sentinel-2
Source: sentinel_2_documentation.pdf
Content: Reference: S2-PDGS-MPC-ATBD-L1 
Issue:  1.1 
Date:   2023-01-31 
9/40 
 
 
           
 
 
Sentinel-2 MSI performs measurements in 13 spectral bands spread over the 
Visible to Near -Infrared (VNIR) and Short -Waves Infrared (SWIR) domains with 
spatial resolutions ranging from 10 to 60 m [REF-11]. 
These spectral channels include: 
• 4 bands at 10 m spatial resolution: blue (490 nm), green (560 n

Result 3
Mission: Sentinel-2
Source: sentinel_2_documentation.pdf
Content: Reference: S2-PDGS-MP